# 第15回　ニューラルネットワーク実装（全結合1層）
***
> **前提**: 第14回 PyTorch 基礎の続きです。第7回では `MyLinearRegression` クラスを自作し `fit` / `predict` を実装しました。PyTorch では `nn.Module` を継承し `forward` メソッドで同様の「入力→出力」の計算を定義します。
>
> | 第7回 `MyLinearRegression` | 第15回 `nn.Module` |
> |---|---|
> | `fit(X, y)` で重みを学習 | 学習ループ + `optimizer.step()` で重みを更新 |
> | `predict(X)` で予測 | `forward(x)` で順伝播 |
> | `score(X, y)` で R² | テストデータで正解率を計算 |

> ⚠️ **この課題で身につけること：コーディングではなく「AI（ニューラルネット）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその処理を選ぶのか**」「**パラメータや特徴量を変えると結果がどう変わるのか**」を理解し、提出物で示すことです。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
>
> コードは原則として完成形ですが、**各問の「核心となる最低限の数行」は `# ★あなたが書く★` として空欄**にしてあります。AI に頼り切らず、要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、設計判断・理由・実験結果・考察を**項目ごとに**記入してください。これが採点対象です。

## 目次
1. SimpleMLP の実装
2. 学習ループ
3. 損失曲線
4. テスト正解率

---

## この回で学ぶこと

### ニューラルネットワークの構造

多層パーセプトロン（MLP：Multilayer Perceptron）は，複数の「全結合層（Linear層）」を重ねたシンプルなニューラルネットワークだ：

```
入力層 (784次元)
   ↓ nn.Linear(784, 128) 重み行列 W₁ (784×128) + バイアス b₁
隠れ層 (128次元)
   ↓ ReLU(x) = max(0, x)  活性化関数
   ↓ nn.Linear(128, 10)  重み行列 W₂ (128×10) + バイアス b₂
出力層 (10次元) ← 数字0〜9のスコア（logits）
```

全部で 784×128 + 128 + 128×10 + 10 = **101,770個のパラメータ**が学習される。

### なぜ活性化関数（ReLU）が必要か

線形層だけを重ねると，どれだけ重ねても「1つの線形変換」と等価になってしまう（行列の積は行列）。**非線形な活性化関数**を挟むことで，複雑な非線形パターンを学習できるようになる。

ReLU（Rectified Linear Unit）: `f(x) = max(0, x)`
- 計算が単純で速い
- **勾配消失問題を回避**（sigmoid や tanh は深いネットで勾配が消えやすかった）
- 現在のデフォルト選択

### 誤差逆伝播法（Backpropagation）の流れ

```
① forward(x)  : 入力から出力を計算（順伝播）
② loss = criterion(output, label)  : 損失を計算
③ optimizer.zero_grad()  : 前のステップの勾配をリセット
④ loss.backward()  : 各パラメータの勾配を計算（逆伝播）
⑤ optimizer.step()  : 勾配を使ってパラメータを更新
```

**③ `zero_grad()` を忘れると**：勾配が前のステップと加算され，誤った更新が起きる。必ずループの最初でリセットすること。

### Adam オプティマイザとは

勾配降下法（SGD）の改良版で，以下の特徴を持つ：
- **各パラメータに個別の学習率**を自動調整
- モーメンタム（過去の勾配の方向を考慮）
- `lr=0.001` が一般的なデフォルト設定

SGD より収束が速く，初心者にも扱いやすいため，現在の標準的な選択肢となっている。

In [ ]:
%pip install -q torch torchvision


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

DATA_ROOT = "./data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


## 問題1　SimpleMLP の構造を読み解く　【説明】
***

### `nn.Module` の仕組み

PyTorch のモデルは `nn.Module` を継承したクラスとして実装する。覚えるべきルールは2つ：

1. `__init__` で使用する層を**定義**する（ここでパラメータが初期化される）
2. `forward` で入力から出力への**計算を記述**する

```python
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()  # ← 必須！親クラスの初期化
        # 層を定義
        self.fc1 = nn.Linear(784, 128)  # 重み行列 (784, 128) を作成
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # 入力 x: (batch_size, 1, 28, 28)
        x = x.view(-1, 784)       # → (batch_size, 784) に flatten
        x = self.relu(self.fc1(x))  # → 線形変換 → ReLU → (batch_size, 128)
        return self.fc2(x)        # → (batch_size, 10) logits を返す
```

### 出力層に softmax/sigmoid がない理由

`CrossEntropyLoss`（問題2で使用）は内部で softmax を計算するため，`forward` の出力は「確率」ではなく「生スコア（logits）」でよい。推論時（`predict_proba`に相当）には `torch.softmax(output, dim=1)` を別途適用する。

### 課題

下のコードセルには `SimpleMLP`（全結合1層の MLP）が **完成形で実装**されています。今回はネットワークを一から書くのではなく、**「なぜこの構造で数字が分類できるのか」を読み解く**のが目的です。

**各行の `# 説明:` の右に、その行が何をしているかを自分の言葉で書いて**ください（コードは変更しない）。書き終えたらセルを実行し、`print(model)` の出力で構造を確認してください。

説明を書くときは、次の問いを意識してください：

- `nn.Linear(28*28, 128)` の `784` と `128` はそれぞれ何の数を表しているか？
- なぜ層と層の間に `ReLU`（非線形）を挟む必要があるのか？ 線形層だけを重ねるとどうなるか？
- `forward` の `x.view(x.size(0), -1)`（flatten）は画像をどんな形に変えているか？
- 出力層が `10` で、しかも softmax を付けていないのはなぜか？（ヒント：問題2の `CrossEntropyLoss`）

In [ ]:
# 完成形の SimpleMLP です。各行の「# 説明:」に自分の言葉で意味を書いてください。
# （コードは変更しない。説明は AI に書かせず自分で書くこと）

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()                       # 説明:
        self.fc1 = nn.Linear(28 * 28, 128)       # 説明:（784 と 128 は何の数？）
        self.relu = nn.ReLU()                    # 説明:（なぜ非線形が必要？）
        self.fc2 = nn.Linear(128, 10)            # 説明:（出力が 10 の理由）

    def forward(self, x):
        x = x.view(x.size(0), -1)                # 説明:（flatten：画像をどんな形に？）
        x = self.relu(self.fc1(x))               # 説明:（線形変換 → 活性化）
        return self.fc2(x)                       # 説明:（logits を返す）


model = SimpleMLP().to(device)
print(model)


In [ ]:
# === ✍️ 問題1 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (1-a) `nn.Linear(28*28, 128)` の `784` と `128` はそれぞれ何を表すか：
1_a_nn_linear_28_28_128 = """
"""

# (1-b) なぜ層と層の間に ReLU（非線形）が必要か（線形層だけを重ねるとどうなるか）
answer_1_b = """
"""

# (1-c) `forward` の `x.view(x.size(0), -1)`（flatten）は画像をどんな形に変えるか
answer_1_c = """
"""

# (1-d) 出力層が 10 で、softmax を付けていない理由
answer_1_d = """
"""



## 問題2　学習ループを読み解く　【説明】
***

### 学習ループの各ステップ解説

```python
for epoch in range(3):          # ← 全データを3周する
    model.train()               # ← 訓練モードに切り替え（Dropout等が有効になる）
    for images, labels in train_loader:   # ← ミニバッチを順に取り出す
        images = images.to(device)        # ← データをGPU/CPUに送る
        labels = labels.to(device)

        optimizer.zero_grad()             # ← ① 勾配をリセット（必須！）
        outputs = model(images)           # ← ② forward: 予測を計算
        loss = criterion(outputs, labels) # ← ③ 損失を計算
        loss.backward()                   # ← ④ backward: 勾配を計算
        optimizer.step()                  # ← ⑤ パラメータを更新
```

### CrossEntropyLoss の意味

多クラス分類の標準的な損失関数だ：
```
CrossEntropyLoss = -Σ y_true × log(softmax(y_pred))
```
- 正解クラスの予測確率が高いほど損失が小さい（0に近づく）
- 損失が下がり続ければ，学習が正しく進んでいることを示す

### `loss.item()` の必要性

`loss` は Tensor であり，計算グラフを保持している。`.item()` を使うと Python の float に変換され，メモリが解放される。ループ内で `.item()` せずに損失を蓄積すると，計算グラフが蓄積されてメモリがあふれる危険がある。

### 課題

下のコードセルには、`SimpleMLP` を MNIST で学習する **完成形の学習ループ**があります（epoch=3, `CrossEntropyLoss`, `Adam(lr=0.001)`）。学習ループは「ニューラルネットがどうやって賢くなるのか」の中心です。

**各行の `# 説明:` の右に、その行が何をしているかを自分の言葉で書いて**ください（コードは変更しない）。書き終えたら実行し、**平均訓練損失がエポックごとに下がる**ことを確認してください。

特に次の3行（誤差逆伝播の中心）は丁寧に説明してください：

- `optimizer.zero_grad()` … なぜ毎ループの先頭で勾配をリセットするのか？ 忘れると何が起きるか？
- `loss.backward()` … ここで何が計算されるのか？（順伝播 forward との違い）
- `optimizer.step()` … `backward()` で求めた何を使って、何を更新するのか？

> **補足**: `total_loss += loss.item()` の `.item()` は何のために必要でしょうか？（ヒント：`loss` は計算グラフを保持した Tensor）

In [ ]:
# 完成形の学習ループです。各行の「# 説明:」に自分の言葉で意味を書いてください。
# （コードは変更しない。説明は自分で書くこと）

criterion = nn.CrossEntropyLoss()                          # 説明:（多クラス分類の損失）
optimizer = optim.Adam(model.parameters(), lr=0.001)       # 説明:（何を更新する？ lr の役割）

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()                                          # 説明:（訓練モードにする意味）
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)  # 説明:（デバイスへ転送）
        optimizer.zero_grad()                              # 説明:（なぜ勾配をリセット？）
        outputs = model(images)                            # 説明:（順伝播 forward）
        loss = criterion(outputs, labels)                  # 説明:（損失を計算）
        loss.backward()                                    # 説明:（逆伝播：何を計算する？）
        optimizer.step()                                   # 説明:（パラメータを更新）
        total_loss += loss.item()                          # 説明:（.item() を使う理由）
    print(f"epoch {epoch + 1}: 平均訓練損失 = {total_loss / len(train_loader):.4f}")


In [ ]:
# === ✍️ 問題2 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (2-a) `optimizer.zero_grad()` を忘れると何が起きるか
answer_2_a = """
"""

# (2-b) `loss.backward()` では何が計算されるか（順伝播 forward との違い）
answer_2_b = """
"""

# (2-c) `optimizer.step()` は backward で求めた何を使い、何を更新するか
answer_2_c = """
"""

# (2-d) `loss.item()` の `.item()` が必要な理由
answer_2_d = """
"""



## 問題3　ハイパーパラメータと損失曲線の実験　【骨格+実験】
***

### 損失曲線（Loss Curve）から何がわかるか

損失曲線はニューラルネットワークの学習状態を診断するための基本ツールだ：

```
【正常な学習】
損失: 2.5 → 1.2 → 0.8 → 0.5 → ...（単調に減少）

【学習率が高すぎる】
損失: 乱高下する，または発散（NaN になる）

【学習率が低すぎる】
損失: ほとんど変化しない

【収束後（過学習が始まると）】
訓練損失: 下がり続ける
検証損失: 下がった後に増加 → これを見たら早期終了（Early Stopping）
```

### 第13回との対応

第13回では決定木の「学習曲線（スコア vs データサイズ）」を描いた。今回は「損失曲線（損失 vs エポック数）」を描く。どちらも「学習の進行状況を可視化する」という目的は同じだ。

### 損失をリストに記録する方法

問題2の学習ループに `epoch_losses` リストを追加して記録する：

```python
epoch_losses = []
for epoch in range(3):
    # ... 学習ループ ...
    avg_loss = total_loss / len(train_loader)
    epoch_losses.append(avg_loss)
```

### 課題

`build_mlp`（隠れユニット数・活性化を変えられる）と `train_and_eval`（学習して損失曲線用のリストと最終 test 精度を返す）が用意されています。ただし、`train_and_eval` 内の **誤差逆伝播の核心4行（`zero_grad` → forward+損失 → `backward` → `step`）はあなたが書きます**（`# ★あなたが書く★`）。問題2で読み解いた内容を、今度は自分の手で書いてみてください。

`EXPERIMENTS` リスト（★印）で **2つ以上の軸**・**最低5通り**を **1回の実行でループ**します。表示される `experiment_log_run` を **✍️ 解答用コードセルの `experiment_log`** に転記し、損失曲線は**最後の1設定**をプロットします。例：

- `LR` を `0.01` / `0.001` / `0.0001` に変える（軸1）
- `HIDDEN` を `32` / `128` / `256` に変える（軸2）
- 余裕があれば `EPOCHS` も `1` / `3` / `5` で比較

> **考察1**: どの設定が最も test 精度が高かったですか？ 損失曲線の「下がり方」と精度の関係を述べてください。
>
> **考察2（損失が下がらない／振動するとき）**: 学習率を大きくしすぎる（例 `0.1`）と損失が振動・発散し、小さくしすぎる（例 `0.00001`）とほとんど下がりません。**なぜ lr が大きすぎても小さすぎてもうまくいかない**のか、解答用コードセルに書いてください。（ヒント：lr は「1ステップで重みをどれだけ動かすか」）


In [ ]:
# === 学習＋損失曲線（完成形）。変えるのは下の ★ の3つの値だけ ===
def build_mlp(hidden=128, activation=nn.ReLU):
    """全結合1層 MLP。隠れユニット数と活性化関数を引数で変えられる。"""
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, hidden),
        activation(),
        nn.Linear(hidden, 10),
    ).to(device)


def train_and_eval(net, epochs=3, lr=0.001):
    """学習して (各エポックの平均損失リスト, 最終 test 精度) を返す。"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr)
    epoch_losses = []
    for epoch in range(epochs):
        net.train()
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            # ★あなたが書く★：誤差逆伝播の核心4ステップ（問題2で読み解いた流れ）
            #   ① 勾配をリセット   ヒント: optimizer.zero_grad()
            #   ② forwardして損失  ヒント: loss = criterion(net(images), labels)
            #   ③ 逆伝播で勾配計算  ヒント: loss.backward()
            #   ④ パラメータ更新    ヒント: optimizer.step()
            ___
            loss = ___
            ___
            ___
            total_loss += loss.item()
        epoch_losses.append(total_loss / len(train_loader))

    net.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = net(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return epoch_losses, correct / total


def describe_loss_curve(epoch_losses):
    if len(epoch_losses) < 2:
        return "データ不足"
    if any(x != x for x in epoch_losses):  # NaN
        return "発散/NaN"
    if epoch_losses[-1] > epoch_losses[0] * 0.95:
        return "ほぼ横ばい"
    if max(abs(epoch_losses[i + 1] - epoch_losses[i]) for i in range(len(epoch_losses) - 1)) > 0.3:
        return "振動/不安定"
    return "順調に減少"


# === ★ここを変えて実験する★：設定の一覧（tag は識別用）===
EXPERIMENTS = [
    {"tag": "baseline", "LR": 0.001, "EPOCHS": 3, "HIDDEN": 128},
    {"tag": "lr_high", "LR": 0.01, "EPOCHS": 3, "HIDDEN": 128},
    {"tag": "lr_low", "LR": 0.0001, "EPOCHS": 3, "HIDDEN": 128},
    {"tag": "hidden_small", "LR": 0.001, "EPOCHS": 3, "HIDDEN": 32},
    {"tag": "hidden_large", "LR": 0.001, "EPOCHS": 3, "HIDDEN": 256},
]

experiment_log_run = []
last_losses = None
last_cfg = None
for cfg in EXPERIMENTS:
    net = build_mlp(hidden=cfg["HIDDEN"])
    epoch_losses, test_acc = train_and_eval(net, epochs=cfg["EPOCHS"], lr=cfg["LR"])
    shape = describe_loss_curve(epoch_losses)
    experiment_log_run.append({
        "tag": cfg["tag"],
        "LR": cfg["LR"],
        "EPOCHS": cfg["EPOCHS"],
        "HIDDEN": cfg["HIDDEN"],
        "最終_test_精度": round(test_acc, 4),
        "損失曲線の形": shape,
    })
    print(f"{cfg['tag']:14s} lr={cfg['LR']:<8} hidden={cfg['HIDDEN']:<4} -> test={test_acc:.4f}  損失={shape}")
    last_losses, last_cfg = epoch_losses, cfg

print("\n--- experiment_log_run（解答欄に転記）---")
import pandas as pd
print(pd.DataFrame(experiment_log_run))

# 損失曲線（最後の設定のみ描画）
plt.plot(range(1, last_cfg["EPOCHS"] + 1), last_losses, marker="o")
plt.xlabel("epoch")
plt.ylabel("train loss")
plt.title(f"損失曲線 ({last_cfg['tag']}: lr={last_cfg['LR']}, hidden={last_cfg['HIDDEN']})")
plt.grid(True)
plt.show()


In [ ]:
# === ✍️ 問題3 解答（採点対象）===
import pandas as pd


# (3-a) 実験ログ（上のセルの experiment_log_run を転記。5通り以上・2軸以上）
experiment_log = pd.DataFrame([
    {'row': 1, 'LR': 0.001, 'EPOCHS': 3, 'HIDDEN': 128, '最終_test_精度': None, '損失曲線の形_下がり方_振動など': None},
    {'row': 2, 'LR': 0.01, 'EPOCHS': 3, 'HIDDEN': 128, '最終_test_精度': None, '損失曲線の形_下がり方_振動など': None},
    {'row': 3, 'LR': 0.0001, 'EPOCHS': 3, 'HIDDEN': 128, '最終_test_精度': None, '損失曲線の形_下がり方_振動など': None},
    {'row': 4, 'LR': 0.001, 'EPOCHS': 3, 'HIDDEN': 32, '最終_test_精度': None, '損失曲線の形_下がり方_振動など': None},
    {'row': 5, 'LR': 0.001, 'EPOCHS': 3, 'HIDDEN': 256, '最終_test_精度': None, '損失曲線の形_下がり方_振動など': None},
    {'row': 6, 'LR': None, 'EPOCHS': None, 'HIDDEN': None, '最終_test_精度': None, '損失曲線の形_下がり方_振動など': None},
])

# (3-b) 考察：最も精度が高かった設定／損失の下がり方と精度の関係
answer_3_b = """
"""

# (3-c) 考察：lr が大きすぎても小さすぎてもうまくいかない理由
answer_3_c = """
"""



## 問題4　活性化関数を選ぶ　【選択】
***

### `model.eval()` と `torch.no_grad()` の意味

推論（テスト）時には，学習時と異なる2点の設定が必要だ：

**① `model.eval()`**：
- Dropout 層を無効化（学習時はランダムにニューロンを無効化するが，推論時は全ニューロンを使う）
- BatchNorm 層の動作を変更（学習時の統計量ではなく，学習済み統計量を使う）
- これを忘れると，推論のたびに結果が変わる（Dropoutのランダム性が残るため）

**② `torch.no_grad()`**：
- 計算グラフを構築しない → **メモリ使用量が激減**
- 推論では勾配計算は不要なため，この最適化が効く
- `with torch.no_grad():` ブロック内では `backward()` が呼べない

### `argmax(dim=1)` の意味

モデルの出力 `outputs` は shape `(batch_size, 10)` の logits（各クラスのスコア）だ。各サンプルについて最も高いスコアのクラス番号を予測値とする：

```python
outputs = [[1.2, -0.5, 3.1, ...],  # スコアが最大は index=2 → 数字「2」と予測
            ...]
preds = outputs.argmax(dim=1)  # → [2, ...]
```

### 活性化関数の選択肢

隠れ層の活性化関数にはいくつか候補があります。代表的な6つ：

| 関数 | 式・概要 | 特徴 |
|---|---|---|
| **(A) ReLU** | `max(0, x)` | 正の入力で勾配が常に 1。計算が速く、深い層でも勾配が消えにくい。現在のデフォルト |
| **(B) Sigmoid** | `1 / (1 + e^(-x))` | 出力 0〜1。入力が大きい/小さいと勾配がほぼ 0 になる（**勾配消失**） |
| **(C) Tanh** | `(eˣ-e⁻ˣ)/(eˣ+e⁻ˣ)` | 出力 -1〜1。Sigmoid より中心化されるが、端では勾配が小さくなる |
| **(D) LeakyReLU** | `x>0 で x, x≤0 で 0.01x` | ReLU の負側を完全に殺さず、わずかな勾配を残す（dying ReLU 対策） |
| **(E) ELU** | `x>0 で x, x≤0 で α(eˣ-1)` | 負側がなめらか。出力平均が 0 に近づき学習が安定しやすい |
| **(F) GELU** | `x·Φ(x)`（標準正規の累積分布） | Transformer 等で標準。なめらかで ReLU に近い挙動 |

### 課題

下のコードセルで、問題3の `build_mlp` の活性化関数を **(A)〜(F) から1つ選んで** `ACTIVATION` に設定し、学習して最終 test 精度を確認してください（`★` の1行だけ変更）。余裕があれば複数試して精度を比べてください。

> **設計判断**: あなたが選んだ活性化関数と、その **理由**を解答用コードセルに書いてください。特に「**なぜ深い層では ReLU が Sigmoid より有利**なのか」を、**勾配消失（gradient vanishing）** の観点から説明してください。（ヒント：Sigmoid/Tanh は入力が大きい/小さい領域で勾配がほぼ 0 になり、逆伝播で勾配が層を遡るほど小さくなってしまう）

In [ ]:
# === ★活性化関数を選んで設定する★ ===
# 下の ACTIVATION を選択肢から1つ選んでください：
#   (A) nn.ReLU   (B) nn.Sigmoid   (C) nn.Tanh
#   (D) nn.LeakyReLU   (E) nn.ELU   (F) nn.GELU
ACTIVATION = nn.ReLU   # ← ここを (A)〜(F) のいずれかに変える

# 問題3で定義した build_mlp / train_and_eval を再利用します
net_act = build_mlp(hidden=128, activation=ACTIVATION)
epoch_losses, test_acc = train_and_eval(net_act, epochs=3, lr=0.001)
print(f"活性化={ACTIVATION.__name__}  ->  最終 test 精度 = {test_acc:.4f}")


In [ ]:
# === ✍️ 問題4 解答（採点対象）===

# (4-a) 選んだ活性化関数（A〜F）：(　)　関数名
# (A) ReLU
# (B) Sigmoid
# (C) Tanh
# (D) LeakyReLU
# (E) GELU
# (F) その他
answer_4_a = ""

# (4-b) その設定での最終 test 精度
answer_4_b = """
"""

# (4-c) 選んだ理由
answer_4_c = """
"""

# (4-d) なぜ深い層では ReLU が Sigmoid より有利か（勾配消失の観点で）
answer_4_d = """
"""

